# Embeddings Qwen3-Embedding (0.6B, 4B e 8B) — TA, C e TAC (Colab A100 ou H200)

Notebook final usado para gerar **todos os vetores Qwen** do projeto, nas duas amostras e nos três modelos
**Qwen3-Embedding-0.6B, 4B e 8B** (célula 3: `MODELOS`, `VERSOES`, `COLUNA_ID`, `LINHAS_ESPERADAS`).

Versões de texto (célula 3, `VERSOES`):
- **TA:** `Título: {titulo}\nResumo: {resumo}`;
- **C:** `Reivindicações: {reivindicacoes}`;
- **TAC:** texto concatenado de título, resumo e reivindicações (só referência; o "TAC" do classificador é a
  **média dos vetores TA e C**, normalizada, como no Gemma 2 — ver `classificador_ipc_qwen8b.ipynb`).

Detalhes que importam:
1. **Limite de 8.192 tokens** (o mesmo das reivindicações no Gemma 2).
2. **O corte preserva o token final `<|endoftext|>` (id 151643)**: o Qwen3 usa o vetor do último token, e o corte
   simples `x[:MAX_TOKENS]` removia esse token justamente nos pedidos truncados, que passavam a ter vetores ruins.
   (Os vetores TAC antigos, cortados em 4.096 sem esse cuidado, desabam nos pedidos longos.)
3. **Pooling pelo último token, padding à esquerda, bf16**; vetores L2-normalizados.
4. **Lotes por número de tokens**, textos de cada bloco ordenados por tamanho, divisão automática do lote se faltar
   memória, e retomada por blocos.

**Dois conjuntos de entrada:**
- **Amostra 1** (PoC 1, pedidos de 2026, 20.697 linhas): `COLUNA_ID = "numero_pedido"`,
  `LINHAS_ESPERADAS = 20697`;
- **Amostra 2** (`dataset_ipc_v2.csv`, 25.563 linhas): `COLUNA_ID = "numero_base"`,
  `LINHAS_ESPERADAS = 25563`.

**Preparação:** coloque o CSV (ou `.csv.gz`) no Google Drive ou envie-o pelo upload da célula 2.
A saída fica em `PASTA_SAIDA/<modelo>/<versão>/`, em blocos: `embeddings_00000.npy`, `pedidos_00000.csv`,
`bloco_00000.json`, `config.json`, `resultado.json`. Os vetores gerados estão no Drive (ver README do repositório).


In [ ]:
#@title 1. Conferir GPU
import torch
assert torch.cuda.is_available(), "Selecione uma GPU (A100 ou H200) antes de continuar."
gpu_name = torch.cuda.get_device_name(0)
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 2**30
print(gpu_name, "| memória (GB):", round(GPU_GB, 1))
if not any(x in gpu_name.upper() for x in ("A100", "H100", "H200")):
    print("Aviso: notebook preparado para A100/H200; reduza TOKENS_POR_LOTE se faltar memória.")

In [ ]:
%pip -q install -U "transformers>=5.0,<6" "huggingface-hub>=1.23,<2" "accelerate>=1.0,<2" "safetensors>=0.8" "tqdm>=4.66"

In [ ]:
#@title 3. Montar Drive e selecionar execução
EM_COLAB = True #@param {type:"boolean"}
if EM_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

CSV_ENTRADA = "/content/drive/MyDrive/PatenteIA/amostra_xml_2026/patentes_xml_embeddings_ipc.csv" #@param {type:"string"}
PASTA_SAIDA = "/content/drive/MyDrive/PatenteIA/embeddings_amostra_xml" #@param {type:"string"}
MODELOS = "todos" #@param ["0.6B", "4B", "8B", "todos"]
VERSOES = "C" #@param ["C", "TA+C", "TA", "TAC"]
MAX_TOKENS = 8192 #@param {type:"integer"}
TAMANHO_BLOCO = 128 #@param {type:"integer"}
TOKENS_POR_LOTE = 0 #@param {type:"integer"}
COLUNA_ID = "numero_pedido" #@param ["numero_pedido", "numero_base"]
LINHAS_ESPERADAS = 20697 #@param {type:"integer"}
# Amostra 1 (PoC 1):        patentes_xml_embeddings_ipc.csv, COLUNA_ID="numero_pedido", LINHAS_ESPERADAS=20697
# Amostra 2 (dataset v2):   dataset_ipc_v2.csv,            COLUNA_ID="numero_base",   LINHAS_ESPERADAS=25563
# Cada conjunto precisa de uma PASTA_SAIDA própria (a retomada confere o CSV pelo SHA-256).
# TOKENS_POR_LOTE=0 usa o padrão por modelo e GPU (soma de tokens com preenchimento por lote);
# em falta de memória, o lote é dividido automaticamente.

from pathlib import Path
assert Path(CSV_ENTRADA).is_file(), f"CSV não encontrado: {CSV_ENTRADA}"
assert 128 <= MAX_TOKENS <= 32768 and TAMANHO_BLOCO > 0 and TOKENS_POR_LOTE >= 0
Path(PASTA_SAIDA).mkdir(parents=True, exist_ok=True)
print("Entrada:", CSV_ENTRADA); print("Saída:", PASTA_SAIDA)

In [ ]:
#@title 4. Conferir CSV e copiar uma vez para o disco local
import csv, hashlib, shutil, sys, unicodedata
csv.field_size_limit(sys.maxsize)
local_csv = Path('/content' if EM_COLAB else '.') / Path(CSV_ENTRADA).name
source_hash = hashlib.sha256()
if Path(CSV_ENTRADA).resolve() == local_csv.resolve():   # arquivo já está no disco local: só calcula o hash
    with local_csv.open('rb') as inp:
        while chunk := inp.read(8 * 1024 * 1024):
            source_hash.update(chunk)
else:
    with Path(CSV_ENTRADA).open('rb') as inp, local_csv.open('wb') as out:
        while chunk := inp.read(8 * 1024 * 1024):
            source_hash.update(chunk); out.write(chunk)
CSV_SHA256 = source_hash.hexdigest()

def clean(value):
    return ' '.join(unicodedata.normalize('NFC', value or '').replace('\\N', '').split())

records = []
with local_csv.open('r', encoding='utf-8-sig', newline='') as inp:
    reader = csv.DictReader(inp)
    assert {COLUNA_ID, 'titulo', 'resumo', 'reivindicacoes'} <= set(reader.fieldnames or []), reader.fieldnames
    seen = set()
    for row in reader:
        pedido = (row[COLUNA_ID] or '').strip()
        assert pedido and pedido not in seen, f"Pedido vazio/duplicado: {pedido!r}"
        seen.add(pedido)
        titulo, resumo, reivindicacoes = (clean(row[k]) for k in ('titulo', 'resumo', 'reivindicacoes'))
        assert resumo and reivindicacoes, f"Resumo/reivindicações ausentes: {pedido}"
        records.append((pedido, titulo, resumo, reivindicacoes))
assert not LINHAS_ESPERADAS or len(records) == LINHAS_ESPERADAS, f"Esperadas {LINHAS_ESPERADAS} linhas; CSV contém {len(records)}"
print(f"{len(records):,} pedidos | {sum(not r[1] for r in records)} sem título | SHA-256 {CSV_SHA256}")

In [ ]:
#@title 5. Funções de geração, retomada e validação de blocos
import gc, io, json, math, os, time
import numpy as np
import torch.nn.functional as F
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer, __version__ as transformers_version

MODEL_IDS = {'0.6B': 'Qwen/Qwen3-Embedding-0.6B', '4B': 'Qwen/Qwen3-Embedding-4B', '8B': 'Qwen/Qwen3-Embedding-8B'}
MODEL_DIMS = {'0.6B': 1024, '4B': 2560, '8B': 4096}
# tokens (com preenchimento) por lote; valores conservadores para A100 40 GB, dobrados em GPUs maiores
LOTE_PADRAO = {'0.6B': 65536, '4B': 24576, '8B': 16384}
TEXT_TEMPLATE = 'patente_titulo_resumo_reivindicacoes_v1'

def document_text(row, version):
    _, title, abstract, claims = row
    if version == 'C':
        return 'Reivindicações: ' + claims
    parts = ['Título: ' + title] if title else []
    parts.append('Resumo: ' + abstract)
    if version == 'TAC':
        parts.append('Reivindicações: ' + claims)
    return '\n'.join(parts)

def special_suffix(tokenizer):
    """Tokens especiais que o tokenizador acrescenta ao fim do texto (ex.: <|endoftext|>)."""
    com = tokenizer('teste de sufixo', add_special_tokens=True)['input_ids']
    sem = tokenizer('teste de sufixo', add_special_tokens=False)['input_ids']
    for i in range(len(com) - len(sem) + 1):
        if com[i:i + len(sem)] == sem:
            return com[i + len(sem):]
    raise ValueError(f'tokenização inesperada: {com} / {sem}')

def atomic_write(path, data):
    temp = path.with_name(path.name + '.tmp')
    with temp.open('wb') as out:
        out.write(data); out.flush(); os.fsync(out.fileno())
    os.replace(temp, path)

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as inp:
        while chunk := inp.read(1024 * 1024):
            digest.update(chunk)
    return digest.hexdigest()

def block_paths(folder, block):
    stem = f'{block:05d}'
    return (folder / f'embeddings_{stem}.npy', folder / f'pedidos_{stem}.csv', folder / f'bloco_{stem}.json')

def valid_block(folder, block, expected_ids, dim):
    embeddings_path, ids_path, done_path = block_paths(folder, block)
    if not all(p.exists() for p in (embeddings_path, ids_path, done_path)):
        return False
    try:
        meta = json.loads(done_path.read_text(encoding='utf-8'))
        if meta['rows'] != len(expected_ids) or meta['dimension'] != dim:
            return False
        if meta['embeddings_sha256'] != sha256_file(embeddings_path) or meta['ids_sha256'] != sha256_file(ids_path):
            return False
        array = np.load(embeddings_path, mmap_mode='r', allow_pickle=False)
        if array.shape != (len(expected_ids), dim) or array.dtype != np.float16:
            return False
        with ids_path.open(encoding='utf-8', newline='') as inp:
            saved_ids = [r['numero_pedido'] for r in csv.DictReader(inp)]
        return saved_ids == expected_ids
    except (OSError, KeyError, ValueError, json.JSONDecodeError):
        return False

def tokenize_clip(texts, tokenizer, max_tokens, suffix):
    """Tokeniza sem truncar (para contar) e corta no limite mantendo o sufixo especial no fim."""
    ids = tokenizer(texts, add_special_tokens=True, truncation=False, padding=False)['input_ids']
    out, truncated = [], 0
    for x in ids:
        if len(x) > max_tokens:
            truncated += 1
            x = x[:max_tokens - len(suffix)] + suffix if suffix else x[:max_tokens]
        out.append(x)
    return out, truncated

def encode_ids(id_lists, tokenizer, model):
    inputs = tokenizer.pad({'input_ids': id_lists}, padding=True, return_tensors='pt').to('cuda')
    try:
        with torch.inference_mode():
            outputs = model(**inputs)
            # padding à esquerda: o último token válido fica na última posição
            last = outputs.last_hidden_state[:, -1].float()
            return F.normalize(last, p=2, dim=1).cpu().numpy().astype(np.float16)
    finally:
        del inputs

def encode_adaptive(id_lists, tokenizer, model):
    try:
        return encode_ids(id_lists, tokenizer, model)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        if len(id_lists) == 1:
            raise RuntimeError('Sem memória até para um texto. Reduza MAX_TOKENS e use uma pasta nova.')
        m = len(id_lists) // 2
        return np.concatenate((encode_adaptive(id_lists[:m], tokenizer, model),
                               encode_adaptive(id_lists[m:], tokenizer, model)))

def encode_block(texts, tokenizer, model, max_tokens, suffix, budget):
    id_lists, truncated = tokenize_clip(texts, tokenizer, max_tokens, suffix)
    order = sorted(range(len(id_lists)), key=lambda i: -len(id_lists[i]))
    vectors = [None] * len(id_lists)
    i = 0
    while i < len(order):
        longest = len(id_lists[order[i]])
        size = max(1, budget // longest)
        sel = order[i:i + size]
        for j, v in zip(sel, encode_adaptive([id_lists[j] for j in sel], tokenizer, model)):
            vectors[j] = v
        i += len(sel)
        if longest > 4096:
            torch.cuda.empty_cache()
    return np.stack(vectors), truncated

def save_block(folder, block, ids, vectors, truncated):
    embeddings_path, ids_path, done_path = block_paths(folder, block)
    buf = io.BytesIO(); np.save(buf, vectors, allow_pickle=False); atomic_write(embeddings_path, buf.getvalue())
    text = io.StringIO(newline=''); w = csv.writer(text); w.writerow(['numero_pedido']); w.writerows((p,) for p in ids)
    atomic_write(ids_path, text.getvalue().encode('utf-8'))
    meta = {'block': block, 'rows': len(ids), 'dimension': int(vectors.shape[1]), 'truncated_rows': truncated,
            'embeddings_sha256': sha256_file(embeddings_path), 'ids_sha256': sha256_file(ids_path)}
    atomic_write(done_path, (json.dumps(meta, ensure_ascii=False, indent=2) + '\n').encode('utf-8'))

def run_combination(model_key, version, tokenizer, model, suffix):
    model_id, dim = MODEL_IDS[model_key], MODEL_DIMS[model_key]
    folder = Path(PASTA_SAIDA) / model_key.replace('.', '_') / version
    folder.mkdir(parents=True, exist_ok=True)
    config = {'csv_sha256': CSV_SHA256, 'rows': len(records), 'model': model_id,
              'model_revision': getattr(model.config, '_commit_hash', None), 'version': version,
              'text_template': TEXT_TEMPLATE, 'max_tokens': MAX_TOKENS, 'sufixo_especial_preservado': suffix,
              'block_size': TAMANHO_BLOCO, 'dimension': dim, 'dtype': 'float16', 'normalized': True,
              'transformers_version': transformers_version}
    config_path = folder / 'config.json'
    if config_path.exists():
        saved = json.loads(config_path.read_text(encoding='utf-8'))
        assert saved == config, f'Parâmetros/CSV diferentes da execução anterior em {folder}. Use outra pasta de saída.'
    else:
        assert not list(folder.glob('bloco_*.json')), f'Blocos sem config em {folder}'
        atomic_write(config_path, (json.dumps(config, ensure_ascii=False, indent=2) + '\n').encode('utf-8'))
    block_count = math.ceil(len(records) / TAMANHO_BLOCO)
    budget = TOKENS_POR_LOTE or LOTE_PADRAO[model_key] * (2 if GPU_GB > 60 else 1)
    print(f'\n{model_id} / {version} -> {folder} | blocos={block_count}, tokens por lote={budget}')
    for block in tqdm(range(block_count), desc=f'{model_key}-{version}'):
        rows = records[block * TAMANHO_BLOCO:(block + 1) * TAMANHO_BLOCO]
        ids = [r[0] for r in rows]
        if valid_block(folder, block, ids, dim):
            continue
        vectors, truncated = encode_block([document_text(r, version) for r in rows], tokenizer, model,
                                          MAX_TOKENS, suffix, budget)
        assert vectors.shape == (len(rows), dim)
        save_block(folder, block, ids, vectors, truncated)
    completed, truncated = 0, 0
    for block in range(block_count):
        ids = [r[0] for r in records[block * TAMANHO_BLOCO:(block + 1) * TAMANHO_BLOCO]]
        assert valid_block(folder, block, ids, dim), f'Bloco ausente/inválido: {block}'
        completed += len(ids)
        truncated += json.loads(block_paths(folder, block)[2].read_text())['truncated_rows']
    result = {'status': 'completo', **config, 'blocks': block_count, 'completed_rows': completed, 'truncated_rows': truncated}
    atomic_write(folder / 'resultado.json', (json.dumps(result, ensure_ascii=False, indent=2) + '\n').encode('utf-8'))
    print(f'Concluído: {completed:,} vetores de {dim} dimensões; {truncated:,} textos truncados.')
    return folder

In [ ]:
#@title 6. Gerar embeddings selecionados
selected_models = list(MODEL_IDS) if MODELOS == 'todos' else [MODELOS]
selected_versions = {'C': ['C'], 'TA+C': ['TA', 'C'], 'TA': ['TA'], 'TAC': ['TAC']}[VERSOES]
completed_folders = []
for model_key in selected_models:
    model_id = MODEL_IDS[model_key]
    print('Carregando:', model_id)
    tokenizer = AutoTokenizer.from_pretrained(model_id, padding_side='left')
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    suffix = special_suffix(tokenizer)
    print('Token(s) especial(is) no fim do texto:', suffix, tokenizer.convert_ids_to_tokens(suffix) if suffix else '(nenhum)')
    model = AutoModel.from_pretrained(model_id, torch_dtype=torch.bfloat16, attn_implementation='sdpa').to('cuda').eval()
    for version in selected_versions:
        t0 = time.time()
        completed_folders.append(run_combination(model_key, version, tokenizer, model, suffix))
        print(f'{model_key}/{version}: {(time.time() - t0) / 60:.1f} min')
    del model, tokenizer
    gc.collect(); torch.cuda.empty_cache()
print('Pastas concluídas:')
for folder in completed_folders:
    print(folder)

## Saída e uso posterior

Cada pasta `MODELO/VERSAO` tem `config.json`, `resultado.json` e os blocos `embeddings_00000.npy`, `pedidos_00000.csv`,
`bloco_00000.json` etc. As linhas de `pedidos_*.csv` correspondem às linhas do `.npy` do mesmo bloco; os vetores são
L2-normalizados, em `float16`.

Para o classificador IPC (`classificador_ipc_qwen8b.ipynb`), aponte para as pastas TA e C de um modelo: o notebook de
treino tira a média dos dois vetores, como no "TAC" do Gemma 2. Se mudar `MAX_TOKENS`, `TAMANHO_BLOCO` ou o CSV, use
outra pasta de saída (a retomada confere os parâmetros e as somas SHA-256 de cada bloco).